# 07 — Verification Resource

**Engineering statement:** verification becomes scarce when draft length, batch size, and rejection risk compete for target-model capacity.

Notebook 00 introduced confidence-scheduled decoding as a specification flow. Notebook 07 makes the first resource model: speculative decoding is only useful when verification capacity is spent on prefixes that are likely to survive target verification.

The goal is not to reproduce the production DSpark scheduler. The goal is to build a small executable model that shows why a scheduler is needed at all.


## 1. Start here

The notebook studies one transition:

```text
draft tokens
→ target verification
→ accepted prefix
→ rejected suffix
→ wasted batch capacity
→ verification budget
```

The central idea is simple:

> A draft token is valuable only if verifying it increases expected accepted tokens more than it consumes scarce target-model capacity.


In [ ]:
from pathlib import Path
from dataclasses import dataclass, asdict
import json
import math
import os
import shutil
import sys
import textwrap
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Robust path discovery whether running from repo root, notebooks/, or an uploaded notebook copy.
CWD = Path.cwd().resolve()
if (CWD / "src").exists() or (CWD / "notebooks").exists():
    REPO_ROOT = CWD
elif (CWD.parent / "notebooks").exists() or (CWD.parent / "figures").exists():
    REPO_ROOT = CWD.parent
else:
    REPO_ROOT = CWD

NOTEBOOKS = REPO_ROOT / "notebooks"
FIGURES = REPO_ROOT / "figures"
RESULTS = REPO_ROOT / "results" / "07_verification_resource"
SITE = REPO_ROOT / "site"
SITE_REPORTS = SITE / "2606-19348"

for path in [NOTEBOOKS, FIGURES, RESULTS, SITE_REPORTS]:
    path.mkdir(parents=True, exist_ok=True)

print(f"Repo root: {REPO_ROOT}")
print(f"Figures:   {FIGURES}")
print(f"Results:   {RESULTS}")


## 2. Resource variables

Speculative decoding has two kinds of work:

1. **Draft work** proposes candidate tokens.
2. **Target verification work** checks a prefix of those candidates.

Notebook 07 treats target verification as the constrained resource. A longer verified prefix can increase accepted tokens, but it also consumes batch capacity.


In [ ]:
@dataclass(frozen=True)
class VerificationResourceVariables:
    gamma: str
    ell: str
    confidence: str
    survival: str
    accepted_tokens: str
    batch_size: str
    verification_waste: str

variables = VerificationResourceVariables(
    gamma="γ = draft proposal length",
    ell="ℓ = scheduled verification length",
    confidence="c_j = conditional probability token j survives, given previous tokens survive",
    survival="a_j = Π_{i≤j} c_i = prefix survival probability",
    accepted_tokens="τ = 1 + Σ_{j≤ℓ} a_j = expected tokens generated per round",
    batch_size="B = Σ_r (1 + ℓ_r) = target verification batch tokens",
    verification_waste="waste = ℓ - Σ_{j≤ℓ} a_j = expected verified tokens that do not survive",
)

pd.DataFrame([asdict(variables)]).T.rename(columns={0: "meaning"})


## 3. Accepted prefix vs rejected suffix

Speculative verification proceeds from left to right. Once one candidate token is rejected, the remaining suffix is discarded.

This means suffix tokens have two risks:

- their own token may be wrong;
- any earlier token may reject first.

Confidence scheduling therefore evaluates **prefix survival**, not isolated token confidence.


In [ ]:
tokens = list("ABCDEFGH")
conditional_confidence = np.array([0.98, 0.96, 0.91, 0.84, 0.62, 0.44, 0.28, 0.18])
prefix_survival = np.cumprod(conditional_confidence)
accept_threshold = 0.50
scheduled_len = int(np.sum(prefix_survival >= accept_threshold))

prefix_df = pd.DataFrame({
    "position": np.arange(1, len(tokens) + 1),
    "token": tokens,
    "conditional_confidence_cj": conditional_confidence,
    "prefix_survival_aj": prefix_survival,
    "scheduled_for_verification": np.arange(1, len(tokens)+1) <= scheduled_len,
})
prefix_df.to_csv(RESULTS / "07_prefix_survival_table.csv", index=False)
prefix_df


In [ ]:
fig, ax = plt.subplots(figsize=(10, 2.7))
ax.set_xlim(-0.6, len(tokens) - 0.4)
ax.set_ylim(0, 1)
ax.axis("off")

for i, (tok, c, a) in enumerate(zip(tokens, conditional_confidence, prefix_survival)):
    scheduled = i < scheduled_len
    hatch = None if scheduled else "//"
    alpha = 1.0 if scheduled else 0.35
    rect = plt.Rectangle((i - 0.38, 0.34), 0.76, 0.32, fill=True, alpha=alpha, hatch=hatch, linewidth=1.4, edgecolor="black", facecolor="white")
    ax.add_patch(rect)
    ax.text(i, 0.52, tok, ha="center", va="center", fontsize=14, fontweight="bold")
    ax.text(i, 0.22, f"c={c:.2f}", ha="center", va="center", fontsize=9)
    ax.text(i, 0.08, f"a={a:.2f}", ha="center", va="center", fontsize=9)

ax.annotate("scheduled prefix", xy=((scheduled_len-1)/2, 0.78), xytext=((scheduled_len-1)/2, 0.96),
            ha="center", arrowprops=dict(arrowstyle="->", linewidth=1.4), fontsize=12)
ax.annotate("deferred suffix", xy=(scheduled_len + 1.0, 0.78), xytext=(scheduled_len + 1.0, 0.96),
            ha="center", arrowprops=dict(arrowstyle="->", linewidth=1.4), fontsize=12)
ax.set_title("Accepted prefix vs. rejected suffix risk", fontsize=15)

fig.tight_layout()
accept_reject_fig = FIGURES / "07_accept_reject_prefix.png"
fig.savefig(accept_reject_fig, dpi=180, bbox_inches="tight")
plt.show()
accept_reject_fig


## 4. Verification budget

A fixed speculative decoder verifies the full draft block. That is simple, but it spends target-model capacity on suffix positions whose expected survival may already be low.

A confidence-scheduled decoder uses the same draft block differently:

```text
fixed verification:       verify all candidates
confidence scheduling:    verify the highest-value prefix
```

The model below separates useful verification from expected waste.


In [ ]:
def expected_accepts(survival, ell):
    return 1.0 + float(np.sum(survival[:ell]))

def verification_waste(survival, ell):
    return float(ell - np.sum(survival[:ell]))

rows = []
for ell in range(0, len(tokens) + 1):
    rows.append({
        "verified_prefix_length_ell": ell,
        "expected_accepted_tokens_tau": expected_accepts(prefix_survival, ell),
        "expected_surviving_draft_tokens": float(np.sum(prefix_survival[:ell])),
        "expected_verification_waste": verification_waste(prefix_survival, ell),
    })
budget_df = pd.DataFrame(rows)
budget_df.to_csv(RESULTS / "07_verification_budget_curve.csv", index=False)
budget_df


In [ ]:
full_ell = len(tokens)
scheduled_ell = scheduled_len
labels = ["Fixed full-block\nverification", "Confidence-scheduled\nverification"]
useful = [np.sum(prefix_survival[:full_ell]), np.sum(prefix_survival[:scheduled_ell])]
waste = [verification_waste(prefix_survival, full_ell), verification_waste(prefix_survival, scheduled_ell)]

fig, ax = plt.subplots(figsize=(8, 4.8))
x = np.arange(len(labels))
ax.bar(x, useful, label="expected surviving draft tokens")
ax.bar(x, waste, bottom=useful, label="expected verification waste")
ax.set_xticks(x, labels)
ax.set_ylabel("Verified draft tokens per round")
ax.set_title("Verification budget: useful prefix vs. expected waste")
ax.legend(loc="upper right")
for i, (u, w) in enumerate(zip(useful, waste)):
    ax.text(i, u / 2, f"useful\n{u:.2f}", ha="center", va="center", fontsize=10)
    ax.text(i, u + w / 2, f"waste\n{w:.2f}", ha="center", va="center", fontsize=10)
fig.tight_layout()

budget_fig = FIGURES / "07_verification_budget_allocation.png"
fig.savefig(budget_fig, dpi=180, bbox_inches="tight")
plt.show()
budget_fig


## 5. Batch capacity under load

Under light load, verifying a few extra suffix tokens may be almost free. Under high load, every extra verification token competes with other active requests for target-model batch capacity.

Notebook 07 uses a simple throughput profile:


a larger verification batch initially improves parallelism, but too many low-value verification tokens reduce steps per second.


In [ ]:
def steps_per_second(batch_tokens, load="medium"):
    """Illustrative target-engine throughput profile.

    This is a compact teaching model, not a production hardware profile.
    """
    batch_tokens = np.asarray(batch_tokens, dtype=float)
    if load == "low":
        base, penalty, knee = 58.0, 0.020, 75.0
    elif load == "medium":
        base, penalty, knee = 48.0, 0.045, 46.0
    elif load == "high":
        base, penalty, knee = 38.0, 0.090, 25.0
    else:
        raise ValueError(load)
    return base / (1.0 + penalty * np.maximum(batch_tokens - knee, 0.0))

R = 24  # active requests
ell_values = np.arange(0, len(tokens) + 1)
load_rows = []
for load in ["low", "medium", "high"]:
    for ell in ell_values:
        B = R * (1 + ell)
        tau = expected_accepts(prefix_survival, ell)
        theta = R * tau * steps_per_second(B, load=load)
        load_rows.append({"load": load, "ell": ell, "batch_tokens_B": B, "tau_per_request": tau, "steps_per_second": float(steps_per_second(B, load=load)), "throughput_theta": theta})
load_df = pd.DataFrame(load_rows)
load_df.to_csv(RESULTS / "07_load_pressure_table.csv", index=False)
load_df.head(12)


In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5))
for load in ["low", "medium", "high"]:
    sub = load_df[load_df["load"] == load]
    ax.plot(sub["ell"], sub["throughput_theta"], marker="o", label=f"{load} load")
    best = sub.loc[sub["throughput_theta"].idxmax()]
    ax.scatter([best["ell"]], [best["throughput_theta"]], s=90, zorder=4)
    ax.text(best["ell"] + 0.05, best["throughput_theta"], f"best ℓ={int(best['ell'])}", fontsize=9, va="center")
ax.set_xlabel("scheduled verification length ℓ")
ax.set_ylabel("illustrative throughput Θ")
ax.set_title("The best verification length changes with system load")
ax.legend()
fig.tight_layout()

load_fig = FIGURES / "07_load_pressure.png"
fig.savefig(load_fig, dpi=180, bbox_inches="tight")
plt.show()
load_fig


## 6. Request mix: not every prefix deserves the same budget

The DSpark paper emphasizes that structured requests such as code and math often sustain higher acceptance than open-ended chat. Notebook 07 turns that observation into a scheduling example.

The scheduler should not ask:

> How long is the draft block?

It should ask:

> Which prefixes have enough survival probability to justify verification under the current load?


In [ ]:
request_profiles = {
    "open chat": np.array([0.90, 0.78, 0.66, 0.54, 0.42, 0.31, 0.23, 0.17]),
    "code": np.array([0.98, 0.96, 0.93, 0.89, 0.84, 0.78, 0.70, 0.62]),
    "math": np.array([0.97, 0.94, 0.90, 0.85, 0.79, 0.72, 0.64, 0.56]),
    "instruction": np.array([0.95, 0.89, 0.82, 0.73, 0.63, 0.52, 0.42, 0.33]),
}

profile_rows = []
threshold = 0.40
for name, conf in request_profiles.items():
    surv = np.cumprod(conf)
    ell = int(np.sum(surv >= threshold))
    profile_rows.append({
        "request_type": name,
        "scheduled_length_at_threshold": ell,
        "expected_tokens_tau": expected_accepts(surv, ell),
        "full_block_waste": verification_waste(surv, len(conf)),
        "scheduled_waste": verification_waste(surv, ell),
    })
profile_df = pd.DataFrame(profile_rows)
profile_df.to_csv(RESULTS / "07_request_profile_scheduling.csv", index=False)
profile_df


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.2))
pos = np.arange(1, 9)
for name, conf in request_profiles.items():
    surv = np.cumprod(conf)
    ax.plot(pos, surv, marker="o", label=name)
ax.axhline(threshold, linestyle="--", linewidth=1.2, label=f"schedule threshold {threshold:.2f}")
ax.set_xlabel("draft position")
ax.set_ylabel("prefix survival probability")
ax.set_title("Request mix creates different verification budgets")
ax.set_xticks(pos)
ax.legend()
fig.tight_layout()

mix_fig = FIGURES / "07_request_mix_prefix_survival.png"
fig.savefig(mix_fig, dpi=180, bbox_inches="tight")
plt.show()
mix_fig


## 7. Fixed verification vs. scheduled verification

A fixed verifier spends the same length on every request. A scheduled verifier adapts the prefix length to the request profile and current load.

This notebook keeps the rule simple: schedule positions whose prefix survival remains above a threshold. Later notebooks replace this threshold with the hardware-aware objective.


In [ ]:
fixed_ell = 8
comparison_rows = []
for name, conf in request_profiles.items():
    surv = np.cumprod(conf)
    sched_ell = int(np.sum(surv >= threshold))
    comparison_rows.append({
        "request_type": name,
        "fixed_ell": fixed_ell,
        "scheduled_ell": sched_ell,
        "fixed_tau": expected_accepts(surv, fixed_ell),
        "scheduled_tau": expected_accepts(surv, sched_ell),
        "fixed_waste": verification_waste(surv, fixed_ell),
        "scheduled_waste": verification_waste(surv, sched_ell),
    })
comparison_df = pd.DataFrame(comparison_rows)
comparison_df["waste_reduction"] = comparison_df["fixed_waste"] - comparison_df["scheduled_waste"]
comparison_df.to_csv(RESULTS / "07_fixed_vs_scheduled.csv", index=False)
comparison_df


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(comparison_df))
width = 0.36
ax.bar(x - width/2, comparison_df["fixed_waste"], width, label="fixed full-block waste")
ax.bar(x + width/2, comparison_df["scheduled_waste"], width, label="scheduled waste")
ax.set_xticks(x, comparison_df["request_type"], rotation=0)
ax.set_ylabel("expected wasted verification tokens")
ax.set_title("Scheduling reduces low-survival verification waste")
ax.legend()
fig.tight_layout()

waste_fig = FIGURES / "07_verification_waste_comparison.png"
fig.savefig(waste_fig, dpi=180, bbox_inches="tight")
plt.show()
waste_fig


## 8. Resource transition

Notebook 07 ends with the design transition that motivates the rest of the repository:

```text
fixed verification
→ confidence-aware verification
→ hardware-aware scheduling
→ adaptive serving
```

The next notebook, **13 — Confidence Scheduling**, makes the scheduling rule explicit.


In [ ]:
flow_steps = [
    "Fixed\nverification",
    "Confidence-aware\nverification",
    "Hardware-aware\nscheduling",
    "Adaptive\nserving",
]

fig, ax = plt.subplots(figsize=(9.5, 2.8))
ax.axis("off")
for i, label in enumerate(flow_steps):
    ax.text(i, 0.5, label, ha="center", va="center", fontsize=12,
            bbox=dict(boxstyle="round,pad=0.55", facecolor="white", edgecolor="black", linewidth=1.3))
    if i < len(flow_steps)-1:
        ax.annotate("", xy=(i+0.72, 0.5), xytext=(i+0.28, 0.5), arrowprops=dict(arrowstyle="->", linewidth=1.5))
ax.set_xlim(-0.6, len(flow_steps)-0.4)
ax.set_ylim(0, 1)
ax.set_title("Verification resource transition", fontsize=15)
fig.tight_layout()

transition_fig = FIGURES / "07_resource_transition.png"
fig.savefig(transition_fig, dpi=180, bbox_inches="tight")
plt.show()
transition_fig


## 9. Key equations

Expected accepted tokens for one scheduled prefix:

\[
\tau(\ell)=1+\sum_{j=1}^{\ell}a_j
\]

Target verification batch size across active requests:

\[
B=\sum_{r=1}^{R}(1+\ell_r)
\]

Expected verification waste:

\[
\mathrm{waste}(\ell)=\ell-\sum_{j=1}^{\ell}a_j
\]

Plain-language statement:

\[
\text{verification value}=\text{accepted prefix gain}-\text{batch-capacity cost}
\]


## 10. Notebook roadmap

| Notebook | Role |
|---|---|
| 00 Context | Establishes confidence scheduling as the repo specification flow. |
| **07 Verification Resource** | Shows why verification is scarce and schedulable. |
| 13 Confidence Scheduling | Builds the confidence-to-prefix scheduling rule. |
| 17 Semi-Autoregressive Design | Studies how the drafter improves suffix quality. |
| 23 Throughput Objective | Turns the resource model into an optimization objective. |
| 29 Hardware-Aware Scheduling | Adds engine throughput profiles and batch capacity. |
| 37 Operating Regimes | Maps low-, medium-, and high-load regions. |
| 43 Resource Allocation | Generalizes confidence-guided allocation beyond decoding. |
| 49 Adaptive AI Infrastructure | Connects scheduled verification to AI operating systems. |


In [ ]:
roadmap = pd.DataFrame([
    {"notebook": "00_context", "role": "specification flow", "question": "How does confidence scheduling organize scalable serving?"},
    {"notebook": "07_verification_resource", "role": "resource model", "question": "Why does verification become scarce?"},
    {"notebook": "13_confidence_scheduling", "role": "scheduler rule", "question": "How should confidence select a prefix?"},
    {"notebook": "17_semi_autoregression", "role": "drafter quality", "question": "How does light autoregression reduce suffix decay?"},
    {"notebook": "23_throughput_objective", "role": "optimization", "question": "Where is throughput maximized?"},
    {"notebook": "29_hardware_aware_scheduling", "role": "systems profile", "question": "How does hardware load change the best length?"},
    {"notebook": "37_operating_regimes", "role": "design regions", "question": "Which policy is best in each serving regime?"},
])
roadmap_path = RESULTS / "07_notebook_roadmap.csv"
roadmap.to_csv(roadmap_path, index=False)
roadmap


## 11. Save notebook manifest

The manifest records the outputs produced by Notebook 07.


In [ ]:
notebook_manifest = {
    "notebook": "07_verification_resource.ipynb",
    "title": "Verification Resource",
    "engineering_statement": "Verification becomes scarce when draft length, batch size, and rejection risk compete for target-model capacity.",
    "primary_question": "Why does DSpark need a scheduler at all?",
    "outputs": {
        "prefix_survival_table": str((RESULTS / "07_prefix_survival_table.csv").relative_to(REPO_ROOT)),
        "verification_budget_curve": str((RESULTS / "07_verification_budget_curve.csv").relative_to(REPO_ROOT)),
        "load_pressure_table": str((RESULTS / "07_load_pressure_table.csv").relative_to(REPO_ROOT)),
        "request_profile_scheduling": str((RESULTS / "07_request_profile_scheduling.csv").relative_to(REPO_ROOT)),
        "fixed_vs_scheduled": str((RESULTS / "07_fixed_vs_scheduled.csv").relative_to(REPO_ROOT)),
        "roadmap": str(roadmap_path.relative_to(REPO_ROOT)),
        "figures": [
            str(accept_reject_fig.relative_to(REPO_ROOT)),
            str(budget_fig.relative_to(REPO_ROOT)),
            str(load_fig.relative_to(REPO_ROOT)),
            str(mix_fig.relative_to(REPO_ROOT)),
            str(waste_fig.relative_to(REPO_ROOT)),
            str(transition_fig.relative_to(REPO_ROOT)),
        ],
    },
}

manifest_path = RESULTS / "07_verification_resource_manifest.json"
manifest_path.write_text(json.dumps(notebook_manifest, indent=2), encoding="utf-8")
notebook_manifest


## 12. Download artifacts

Run the final cell to package the Notebook 07 outputs for download.


In [ ]:
from IPython.display import FileLink, display

zip_path = RESULTS / "07_verification_resource_artifacts.zip"
notebook_path = NOTEBOOKS / "07_verification_resource.ipynb"
fallback_notebook_path = Path.cwd() / "07_verification_resource.ipynb"

paths_to_include = [
    notebook_path if notebook_path.exists() else fallback_notebook_path,
    FIGURES,
    RESULTS,
]

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for item in paths_to_include:
        item = Path(item)
        if item.is_dir():
            for path in item.rglob("*"):
                if not path.is_file():
                    continue
                if path.resolve() == zip_path.resolve():
                    continue
                try:
                    archive_name = path.relative_to(REPO_ROOT)
                except ValueError:
                    archive_name = path.name
                zf.write(path, archive_name)
        elif item.exists():
            if item.resolve() == zip_path.resolve():
                continue
            try:
                archive_name = item.relative_to(REPO_ROOT)
            except ValueError:
                archive_name = item.name
            zf.write(item, archive_name)

print(f"Created: {zip_path}")
print(f"Size: {zip_path.stat().st_size:,} bytes")
display(FileLink(str(zip_path)))

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    pass
